In [1]:
import pandas as pd

In [2]:
### reading the csv: FrogID5_dataset
filename = "../data/raw/FrogID5_dataset.csv"

# load the relevant columns only (others are either empty, contail all the same information or contain irrelevant information)
columns_to_load_frogID = [
    'occurrenceID', 'eventID', 'decimalLatitude', 'decimalLongitude', 'scientificName',
    'eventDate', 'eventTime', 'coordinateUncertaintyInMeters', 'geoprivacy', 'recordedBy', 'stateProvince'
]

frogID5_database_original = pd.read_csv(filename, usecols=columns_to_load_frogID, sep=',', on_bad_lines='skip')

frogID5_database_edited = frogID5_database_original.rename(columns={'stateProvince': 'stateProvinceShort'})
frogID5_database_edited

,occurrenceID,eventID,decimalLatitude,decimalLongitude,scientificName,eventDate,eventTime,coordinateUncertaintyInMeters,geoprivacy,recordedBy,stateProvinceShort
0,1,4493,-38.1000,144.600,Limnodynastes peronii,2017-11-14,11:49:20UTC,10000.000000,obscured,3817,VIC
1,2,4496,-17.0000,145.700,Litoria rheocola,2017-11-14,12:18:49UTC,10000.000000,obscured,1595,QLD
2,3,4503,-17.0000,145.700,Litoria dayi,2017-11-14,12:43:09UTC,10000.000000,obscured,1595,QLD
3,4,5349,-38.1000,144.600,Limnodynastes peronii,2017-11-18,21:45:45UTC,10000.000000,obscured,4558,VIC
4,5,5603,-38.1000,144.600,Limnodynastes peronii,2017-11-18,11:01:17UTC,10000.000000,obscured,3817,VIC
...,...,...,...,...,...,...,...,...,...,...,...
771537,771538,283438,-31.9335,115.912,Litoria adelaidensis,2019-10-13,19:14:00+0800,899.999023,open,13928,WA
771538,771539,283462,-31.9335,115.912,Litoria adelaidensis,2019-09-28,18:22:31+0800,1000.000000,open,13928,WA
771539,771540,385997,-33.3263,151.448,Litoria gracilenta,2022-02-26,21:31:52+1100,2800.000000,open,8433,NSW
771540,771541,442368,-31.9340,115.914,Litoria adelaidensis,2022-09-10,18:18:32+0800,600.000000,open,13928,WA


In [3]:
### extending the FrogID5_dataset with the vernacular names of the species

## webscraping a table that contains scientific and venacular names of australian amphibia
url = "https://amphibiaweb.org/cgi-bin/amphib_query?query_src=aw_maps_geo-ocea&rel-isocc=like&orderbyaw=Order&where-isocc=Australia"
List_of_amphibia_species_of_Australia  = pd.read_html(url)
amphibia_of_australia = List_of_amphibia_species_of_Australia[1]

## cleaning data: drop rows that only contain NaN, reset_index, choose only important columns, renaming columns
amphibia_of_australia_cleaned = amphibia_of_australia.dropna(how='all').reset_index()
amphibia_of_australia_cleaned = amphibia_of_australia_cleaned[['Scientific Name', 'Vernacular Name', 'Family']]
amphibia_of_australia_cleaned = amphibia_of_australia_cleaned.rename(columns={'Scientific Name': 'scientificName', 'Vernacular Name': 'vernacularName'})

## merging the amphibia dataframe with the FrogID5_dataset
frogID5_database_edited = pd.merge(frogID5_database_edited, amphibia_of_australia_cleaned, on='scientificName', how='left')

In [4]:
### extending the FrogID5_dataset with the full state / territory names

## webscraping a table that contains all australian states and territories and its short forms
url = "https://en.wikipedia.org/wiki/States_and_territories_of_Australia"
States_and_territories_of_Australia = pd.read_html(url, header=0)
states_and_territories = States_and_territories_of_Australia[9]

## cleaning data: dropping irrelevant rows, selecting important columns, renaming columns
indices_to_drop = [0, 9, 10, 11, 12, 13, 14]
states_and_territories = states_and_territories.drop(indices_to_drop).reset_index()[['Postal', 'State/territory']]
states_and_territories = states_and_territories.rename(columns={'Postal': 'stateProvinceShort', 'State/territory': 'stateProvince'})

## merging the stateProvinces dataframe with the FrogID5_dataset
frogID5_database_edited = pd.merge(frogID5_database_edited, states_and_territories, on='stateProvinceShort', how='left')

In [5]:
### extending the FrogID5_dataset with columns for year, month and day of the event date

frogID5_database_edited[['year', 'month', 'day']] = frogID5_database_edited['eventDate'].str.split('-', expand=True)

In [6]:
### saving the edited database as a csv file

frogID5_database_final = frogID5_database_edited
frogID5_database_final.to_csv('../data/processed/frogID5_database_prepared.csv', index=False)